# Akan-BPE Model Ladder - Light Split

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/run-full-light.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/run-full-light.ipynb)

Runs the lighter half of the ladder: Qwen3-0.6B, Llama-3.2-1B, and tiny-aya-base.

Run this top-to-bottom on Kaggle with a 2x T4 GPU accelerator. Model integration now uses the balanced `models/mixed_tokenizer.json` tokenizer instead of the earlier TTS-only tokenizer. Existing output cells may still show the previous TTS-tokenizer run until this notebook is rerun. Each model runs two arms concurrently: random embedding init on GPU 0 and mean-of-subword embedding init on GPU 1. After each model, the notebook prints fertility/tokenization metrics, BPB metrics, generation samples, reload verification, the full JSON payloads, and then zips artifacts into `outputs/`.

## Setup

In [ ]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

# Leave all GPUs visible; run_parallel_arms pins each child process explicitly.
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

In [ ]:
!pip install -q uv
!uv pip install -q --system -e ".[dev,train]"
!uv pip install -q --system bitsandbytes sentencepiece

In [ ]:
# Hugging Face authentication.
# On Kaggle: add your token as a secret named HF_TOKEN (Notebook settings → Secrets).
# Elsewhere: falls back to an interactive prompt.
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    login()


In [ ]:
# Confirm GPUs are available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a 2x T4 GPU accelerator, then re-run.")
print(f"CUDA devices: {torch.cuda.device_count()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

In [ ]:
# Download Akan datasets. --tts-limit 50000 pins the 40,000/5,000/5,000 split.
!python scripts/download.py --output-dir data --tts-limit 50000

In [ ]:
# Verify required inputs exist. The balanced mixed tokenizer is committed under models/.
from pathlib import Path

required = {
    "Twi train data": Path("data/pristine_twi_train.jsonl"),
    "Twi test data": Path("data/pristine_twi_test.jsonl"),
    "Mixed tokenizer": Path("models/mixed_tokenizer.json"),
}
missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")

## Reporting helpers

In [ ]:
import json
from pathlib import Path


def load_result(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing result JSON: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def fmt(value, digits=4):
    if value is None:
        return "n/a"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def result_paths(model_slug):
    return {
        "random": Path("results") / f"run-{model_slug}-mixed.json",
        "mean_subword": Path("results") / f"run-{model_slug}-mixed-meansub.json",
    }


def print_tokenization_table(results):
    print("\nTOKENIZATION / FERTILITY")
    print("tokens/word is fertility; lower is better.")
    header = (
        f"{'arm':<22}{'base fertility':>16}{'Akan fertility':>16}"
        f"{'base tokens':>14}{'Akan tokens':>14}{'words':>10}{'reduction':>12}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        counts = payload["token_count_comparison"]
        base = counts["base_model_tokenizer"]
        akan = counts["experiment_tokenizer"]
        print(
            f"{arm:<22}"
            f"{fmt(base.get('fertility')):>16}"
            f"{fmt(akan.get('fertility')):>16}"
            f"{base.get('total_tokens'):>14}"
            f"{akan.get('total_tokens'):>14}"
            f"{akan.get('total_words'):>10}"
            f"{fmt(counts.get('token_reduction_ratio')):>12}"
        )


def print_bpb_table(results):
    print("\nMODELING / BPB")
    print("BPB uses full byte coverage; lower experiment BPB is better.")
    header = (
        f"{'arm':<22}{'eval_loss':>12}{'perplexity':>12}"
        f"{'experiment BPB':>18}{'base BPB':>12}{'improvement':>14}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        bpb = payload["eval"]["bpb"]
        base_bpb = (bpb.get("base") or {}).get("bits_per_byte")
        print(
            f"{arm:<22}"
            f"{fmt(payload['eval'].get('eval_loss')):>12}"
            f"{fmt(payload['eval'].get('perplexity'), 2):>12}"
            f"{fmt(bpb['experiment'].get('bits_per_byte')):>18}"
            f"{fmt(base_bpb):>12}"
            f"{fmt(bpb.get('improvement')):>14}"
        )


def print_generation_quality_table(results):
    print("\nGENERATION QUALITY / chrF")
    print("Held-out continuation setup: prompt words -> reference words; higher chrF is better.")
    header = (
        f"{'arm':<22}{'examples':>10}{'prompt':>10}{'reference':>12}"
        f"{'max_new':>10}{'chrF':>12}{'chrF++':>12}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        quality = payload.get("eval", {}).get("generation_quality") or {}
        print(
            f"{arm:<22}"
            f"{fmt(quality.get('num_examples')):>10}"
            f"{fmt(quality.get('prompt_words')):>10}"
            f"{fmt(quality.get('reference_words')):>12}"
            f"{fmt(quality.get('max_new_tokens')):>10}"
            f"{fmt(quality.get('chrf')):>12}"
            f"{fmt(quality.get('chrfpp')):>12}"
        )


def print_generation_samples(results):
    print("\nGENERATION SAMPLES")
    for arm, payload in results.items():
        print(f"\n[{arm}]")
        for index, sample in enumerate(payload.get("generation_samples", []), start=1):
            print(f"sample {index} prompt: {sample.get('prompt', '')}")
            print(f"sample {index} completion: {sample.get('completion', '')}")


def print_run_integrity(results):
    print("\nRUN INTEGRITY")
    header = f"{'arm':<22}{'output_model_dir':<34}{'reload ok':>10}{'device':>18}"
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        reload_info = payload.get("reload_verification") or {}
        device = payload.get("device") or {}
        print(
            f"{arm:<22}"
            f"{str(payload.get('output_model_dir')):<34}"
            f"{str(reload_info.get('success')):>10}"
            f"{str(device.get('device_name')):>18}"
        )


def print_full_json(results):
    for arm, payload in results.items():
        print(f"\nBEGIN_FULL_JSON {arm}")
        print(json.dumps(payload, ensure_ascii=False, indent=2))
        print(f"END_FULL_JSON {arm}")


def print_arm_full_json(payload):
    experiment_id = payload.get("experiment_id", "unknown")
    print(f"\nBEGIN_ARM_FULL_JSON {experiment_id}")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    print(f"END_ARM_FULL_JSON {experiment_id}")


def display_arm_result(experiment_id):
    path = Path("results") / f"{experiment_id}.json"
    payload = load_result(path)
    print("=" * 88)
    print(f"ARM RESULT FOR {experiment_id}")
    print("=" * 88)
    print(f"Result file: {path}")
    print_bpb_table({experiment_id: payload})
    print_generation_quality_table({experiment_id: payload})
    print_run_integrity({experiment_id: payload})
    print_arm_full_json(payload)
    return payload


def display_model_results(model_slug):
    paths = result_paths(model_slug)
    results = {arm: load_result(path) for arm, path in paths.items()}
    print("=" * 88)
    print(f"FULL RESULTS FOR {model_slug}")
    print("=" * 88)
    print("Result files:")
    for arm, path in paths.items():
        print(f"- {arm}: {path}")
    print_tokenization_table(results)
    print_bpb_table(results)
    print_generation_quality_table(results)
    print_generation_samples(results)
    print_run_integrity(results)
    best_bpb = min(results, key=lambda arm: results[arm]["eval"]["bpb"]["experiment"]["bits_per_byte"])
    scored = {
        arm: payload.get("eval", {}).get("generation_quality", {}).get("chrf")
        for arm, payload in results.items()
    }
    scored = {arm: score for arm, score in scored.items() if score is not None}
    if scored:
        best_chrf = max(scored, key=scored.get)
        print(f"\nBest generation chrF arm: {best_chrf}")
    print(f"Best downstream BPB arm: {best_bpb}")
    print_full_json(results)
    return results

import os
import subprocess
import sys
import time


def _tail_text(path, lines=40):
    path = Path(path)
    if not path.exists():
        return ""
    text = path.read_text(encoding="utf-8", errors="replace")
    return "\n".join(text.splitlines()[-lines:])


def _arm_command(
    *,
    model_slug,
    model_id,
    embedding_init_mode,
    generation_eval_samples,
    generation_prompt_words,
    generation_reference_words,
    generation_eval_max_new_tokens,
    extra_args,
):
    suffix = "" if embedding_init_mode == "random" else "-meansub"
    experiment_id = f"run-{model_slug}-mixed{suffix}"
    return [
        sys.executable,
        "-u",
        "scripts/model_integration.py",
        "--model-id",
        model_id,
        "--experiment-id",
        experiment_id,
        "--output-dir",
        str(Path("models") / experiment_id),
        "--results-output",
        str(Path("results") / f"{experiment_id}.json"),
        "--tokenizer-path",
        "models/mixed_tokenizer.json",
        "--embedding-init-mode",
        embedding_init_mode,
        "--generation-eval-samples",
        str(generation_eval_samples),
        "--generation-prompt-words",
        str(generation_prompt_words),
        "--generation-reference-words",
        str(generation_reference_words),
        "--generation-eval-max-new-tokens",
        str(generation_eval_max_new_tokens),
        *list(extra_args or []),
    ]


def run_parallel_arms(
    model_slug,
    model_id,
    generation_eval_samples=512,
    generation_prompt_words=48,
    generation_reference_words=64,
    generation_eval_max_new_tokens=64,
    require_two_gpus=True,
    extra_args=None,
):
    import torch

    device_count = torch.cuda.device_count()
    print(f"Detected {device_count} CUDA devices")
    if device_count < 2:
        message = (
            "run_parallel_arms requires at least 2 CUDA GPUs so the random and "
            "mean_subword arms can run concurrently. Switch the Kaggle accelerator "
            "to 2x T4, or call with require_two_gpus=False only for a deliberate "
            "local smoke test."
        )
        if require_two_gpus:
            raise RuntimeError(message)
        print(f"WARNING: {message}")

    Path("logs").mkdir(parents=True, exist_ok=True)
    Path("results").mkdir(parents=True, exist_ok=True)
    Path("models").mkdir(parents=True, exist_ok=True)

    arms = [
        ("random", "0", Path("logs") / f"run-{model_slug}-mixed-random.log"),
        ("mean_subword", "1", Path("logs") / f"run-{model_slug}-mixed-meansub.log"),
    ]
    processes = []
    for embedding_init_mode, cuda_device, log_path in arms:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = cuda_device
        env.setdefault("TOKENIZERS_PARALLELISM", "false")
        env.setdefault("PYTHONUNBUFFERED", "1")
        command = _arm_command(
            model_slug=model_slug,
            model_id=model_id,
            embedding_init_mode=embedding_init_mode,
            generation_eval_samples=generation_eval_samples,
            generation_prompt_words=generation_prompt_words,
            generation_reference_words=generation_reference_words,
            generation_eval_max_new_tokens=generation_eval_max_new_tokens,
            extra_args=extra_args,
        )
        log_file = log_path.open("w", encoding="utf-8")
        print(f"Starting {model_slug}/{embedding_init_mode} on CUDA_VISIBLE_DEVICES={cuda_device}")
        print(f"  log: {log_path}")
        print(f"  command: {' '.join(command)}")
        process = subprocess.Popen(
            command,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            text=True,
        )
        processes.append(
            {
                "arm": embedding_init_mode,
                "cuda_device": cuda_device,
                "log_path": log_path,
                "log_file": log_file,
                "process": process,
            }
        )

    failures = []
    try:
        while True:
            unfinished = [item for item in processes if item["process"].poll() is None]
            if not unfinished:
                break
            status = ", ".join(f"{item['arm']}=running" for item in unfinished)
            print(f"Waiting for {model_slug}: {status}")
            time.sleep(60)
    except KeyboardInterrupt:
        for item in processes:
            if item["process"].poll() is None:
                item["process"].terminate()
        raise
    finally:
        for item in processes:
            item["log_file"].close()

    for item in processes:
        returncode = item["process"].returncode
        arm = item["arm"]
        log_path = item["log_path"]
        print("=" * 88)
        print(f"{model_slug}/{arm} finished with exit code {returncode}")
        print(f"Log tail: {log_path}")
        print(_tail_text(log_path))
        if returncode != 0:
            failures.append(f"{arm} exited {returncode}; see {log_path}")

    if failures:
        raise RuntimeError("Parallel arm run failed: " + "; ".join(failures))

    paths = result_paths(model_slug)
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing expected result file(s): " + ", ".join(missing))
    print(f"Both arms completed for {model_slug}. Result files are ready for display_model_results().")



## Run qwen-0.6b

Model id: `Qwen/Qwen3-0.6B-Base`. This section runs both arms concurrently on 2x T4 when available, then immediately displays and archives the full results.

In [ ]:
run_parallel_arms("qwen-0.6b", "Qwen/Qwen3-0.6B-Base")


## Full results and artifact zip - qwen-0.6b

In [ ]:
results_qwen_0_6b = display_model_results("qwen-0.6b")

In [ ]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-qwen-0.6b-artifacts.zip     results/run-qwen-0.6b.json     results/run-qwen-0.6b-meansub.json     models/run-qwen-0.6b     models/run-qwen-0.6b-meansub
!ls -lh outputs

## Run llama-1b

Model id: `meta-llama/Llama-3.2-1B`. This section runs both arms concurrently on 2x T4 when available, then immediately displays and archives the full results.

In [ ]:
run_parallel_arms("llama-1b", "meta-llama/Llama-3.2-1B")


## Full results and artifact zip - llama-1b

In [ ]:
results_llama_1b = display_model_results("llama-1b")

In [ ]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-llama-1b-artifacts.zip     results/run-llama-1b.json     results/run-llama-1b-meansub.json     models/run-llama-1b     models/run-llama-1b-meansub
!ls -lh outputs

## Run aya-base

Model id: `CohereLabs/tiny-aya-base`. This section runs both arms concurrently on 2x T4 when available, then immediately displays and archives the full results.

In [ ]:
run_parallel_arms("aya-base", "CohereLabs/tiny-aya-base")


## Full results and artifact zip - aya-base

In [ ]:
results_aya_base = display_model_results("aya-base")

In [ ]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-aya-base-artifacts.zip     results/run-aya-base.json     results/run-aya-base-meansub.json     models/run-aya-base     models/run-aya-base-meansub
!ls -lh outputs

## Notebook-level summary

In [ ]:
MODEL_SLUGS = ['qwen-0.6b', 'llama-1b', 'aya-base']
SPLIT_NAME = 'light'
all_results = {slug: {arm: load_result(path) for arm, path in result_paths(slug).items()} for slug in MODEL_SLUGS}
flat_results = {f"{slug}/{arm}": payload for slug, arms in all_results.items() for arm, payload in arms.items()}

print("=" * 88)
print("NOTEBOOK SPLIT SUMMARY")
print("=" * 88)
print_tokenization_table(flat_results)
print_bpb_table(flat_results)
print_generation_quality_table(flat_results)

summary = {
    slug: {
        arm: {
            "eval_loss": payload["eval"]["eval_loss"],
            "perplexity": payload["eval"]["perplexity"],
            "experiment_bpb": payload["eval"]["bpb"]["experiment"]["bits_per_byte"],
            "base_bpb": (payload["eval"]["bpb"].get("base") or {}).get("bits_per_byte"),
            "bpb_improvement": payload["eval"]["bpb"].get("improvement"),
            "base_fertility": payload["token_count_comparison"]["base_model_tokenizer"]["fertility"],
            "akan_fertility": payload["token_count_comparison"]["experiment_tokenizer"]["fertility"],
            "token_reduction_ratio": payload["token_count_comparison"]["token_reduction_ratio"],
            "generation_quality": {
                "chrf": (payload.get("eval", {}).get("generation_quality") or {}).get("chrf"),
                "chrfpp": (payload.get("eval", {}).get("generation_quality") or {}).get("chrfpp"),
                "num_examples": (payload.get("eval", {}).get("generation_quality") or {}).get("num_examples"),
                "prompt_words": (payload.get("eval", {}).get("generation_quality") or {}).get("prompt_words"),
                "reference_words": (payload.get("eval", {}).get("generation_quality") or {}).get("reference_words"),
                "max_new_tokens": (payload.get("eval", {}).get("generation_quality") or {}).get("max_new_tokens"),
            },
        }
        for arm, payload in arms.items()
    }
    for slug, arms in all_results.items()
}
full_payload = {
    "split": SPLIT_NAME,
    "model_slugs": MODEL_SLUGS,
    "summary": summary,
    "runs": all_results,
}
Path("outputs").mkdir(exist_ok=True)
Path(f"outputs/{SPLIT_NAME}-split-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
Path(f"outputs/{SPLIT_NAME}-split-full-results.json").write_text(json.dumps(full_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote outputs/{SPLIT_NAME}-split-summary.json")
print(f"Wrote outputs/{SPLIT_NAME}-split-full-results.json")
print(f"\nBEGIN_NOTEBOOK_FULL_JSON {SPLIT_NAME}")
print(json.dumps(full_payload, ensure_ascii=False, indent=2))
print(f"END_NOTEBOOK_FULL_JSON {SPLIT_NAME}")


In [ ]:
!ls -lh outputs